In [2]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['Inmobiliara'] 
coleccion = db['G2_Inmobiliaria_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [6]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")


# --- CONEXIÓN A MONGODB (TU CONFIGURACIÓN) ---
client = MongoClient('mongodb://mongodb:27017/')
db = client['Inmobiliara']
coleccion = db['G2_Inmobiliaria_scraping']

print("Conexión exitosa a MongoDB")


# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2_Inmobiliaria"
datos_finales = []
driver = None


# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN ---
    limite_paginas = 3

    driver.get("https://www.portalinmobiliario.com/venta/departamento")

    for nivel_pagina in range(limite_paginas):

        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        time.sleep(10)

        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "li.ui-search-layout__item")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR,
            "li.ui-search-layout__item"
        )

        print(f"Propiedades encontradas: {len(bloques)}")

        for bloque in bloques:

            try:
                nombre = bloque.find_element(
                    By.CSS_SELECTOR,
                    "a.poly-component__title"
                ).text

                precio = bloque.find_element(
                    By.CSS_SELECTOR,
                    "span.andes-money-amount__fraction"
                ).text

                ubicacion = bloque.find_element(
                    By.CSS_SELECTOR,
                    "span.poly-component__location"
                ).text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "ubicacion": ubicacion,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "fuente": "PortalInmobiliario"
                })

            except:
                continue


        # --- SIGUIENTE PÁGINA ---
        try:
            btn_sig = driver.find_element(
                By.CSS_SELECTOR,
                "li.andes-pagination__button--next a"
            )

            driver.execute_script(
                "arguments[0].click();",
                btn_sig
            )

            time.sleep(5)

        except:
            print("No se encontró el botón siguiente.")
            break


    print(f" Extracción terminada: {len(datos_finales)} propiedades.")


except Exception as e:
    print(f" Error en Selenium: {e}")


finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass


# --- PASO 3: GUARDAR EN MONGODB ---

try:

    if datos_finales:

        for d in datos_finales:

            v_limpio = str(d["valor"])\
                .replace(".", "")\
                .replace("$", "")\
                .strip()

            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0


        coleccion.insert_many(datos_finales)

        print(" Datos cargados en MongoDB correctamente.")
        print(f" Total guardados: {len(datos_finales)}")

    else:
        print(" No hay datos para guardar.")


except Exception as e:
    print(f" Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
Conexión exitosa a MongoDB
 Navegador iniciado correctamente.
--- Procesando Página 1 ---
Propiedades encontradas: 48
--- Procesando Página 2 ---
Propiedades encontradas: 48
--- Procesando Página 3 ---
Propiedades encontradas: 48
 Extracción terminada: 144 propiedades.
 Error en MongoDB: mongodb:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69df979fdd9f8bd90ae52d5d, topology_type: Unknown, servers: [<ServerDescription ('mongodb', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('mongodb:27017: [Errno -2] Name or service not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
